# Permutation importance op de 64 features

**Taak (lesweek 11):** meten hoeveel elk van de huidige 64 features bijdraagt aan het voorspellen van de klasse, met de SVM uit `hommel_classificatie.ipynb`.

**Aanpak**
1. Dezelfde data-opzet als `hommel_classificatie.ipynb`: dubbele rijen eruit, drie klassen (`buzz`, `flight`, `ruis`).
2. Dezelfde testset apart houden (eerste split van `StratifiedGroupKFold`, `SEED = 42`). Volgens de afspraken in `docs/planning.md` gebruiken we de testset pas in lesweek 16. Alles hieronder gebeurt dus alleen op de **trainingsset**.
3. Op de trainingsset: 5-voudige kruisvalidatie met `StratifiedGroupKFold` op `source_file`. Alle seconden uit één opname zitten dus samen in de train- óf de validatiefold. Seconden uit dezelfde opname lijken sterk op elkaar; splitsen op losse rijen zou de importances te rooskleurig maken.
4. Per fold: SVM trainen, daarna `permutation_importance` berekenen op de **validatiefold** (data die het model niet gezien heeft).
5. Resultaat: per feature het gemiddelde en de spreiding (std) van de importance over de folds.

**Buiten scope (lesweek 12):** features weglaten en opnieuw trainen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 42
KLASSEN = ["buzz", "flight", "ruis"]
FEATURES_PAD = "../data/processed_data/bee_features_per_second.csv"

N_SPLITS = 5          # folds binnen de trainingsset
N_REPEATS = 10        # hoe vaak elke feature per fold door elkaar gehusseld wordt
SCORING = "f1_macro"  # macro-F1: buzz en flight tellen even zwaar als ruis

# Dezelfde SVM als in hommel_classificatie.ipynb (standaardinstellingen: C=1, gamma="scale")
svm = make_pipeline(StandardScaler(), SVC(class_weight="balanced"))

## 1. Data laden

Zelfde opschoning als in `hommel_classificatie.ipynb`. De dubbele rijen halen we hier zelf weg; in lesweek 12 wordt `preprocessdata.py` gerepareerd zodat ze er niet meer in komen.

In [ ]:
features = pd.read_csv(FEATURES_PAD)
FEATURES = [c for c in features.columns if c not in ["start", "end", "label", "source_file"]]
assert len(FEATURES) == 64, f"verwacht 64 features, gevonden {len(FEATURES)}"

print(f"Rijen in de CSV: {len(features)}")
print(f"Dubbele rijen (zelfde bestand + zelfde seconde): {features.duplicated(subset=['source_file', 'start']).sum()}")

df = features.drop_duplicates(subset=["source_file", "start"]).copy()
df = df.sort_values(["source_file", "start"]).reset_index(drop=True)

# drie klassen: buzz, flight, ruis (= radio + voice + noise)
df["klasse"] = df["label"].map({"buzz": "buzz", "flight": "flight"})
df.loc[df["label"].notna() & df["klasse"].isna(), "klasse"] = "ruis"

lab = df[df["klasse"].notna()].reset_index(drop=True)
print(f"Gelabelde seconden: {len(lab)} uit {lab['source_file'].nunique()} opnames")
lab["klasse"].value_counts()

## 2. Testset apart houden

Precies dezelfde split als in `hommel_classificatie.ipynb`, zodat de testset van het team onaangeroerd blijft.

In [ ]:
X = lab[FEATURES]
y = lab["klasse"].values
groepen = lab["source_file"].values

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groepen))

X_train, y_train, g_train = X.iloc[train_idx].reset_index(drop=True), y[train_idx], groepen[train_idx]
assert set(g_train).isdisjoint(groepen[test_idx])

print(f"Trainingsset: {len(y_train)} seconden uit {len(set(g_train))} opnames  (hier rekenen we mee)")
print(f"Testset:      {len(test_idx)} seconden uit {len(set(groepen[test_idx]))} opnames  (niet gebruikt, pas in lesweek 16)")

## 3. Folds binnen de trainingsset

Elke validatiefold moet alle drie de klassen bevatten, anders is de macro-F1 per fold niet vergelijkbaar.

In [ ]:
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(cv.split(X_train, y_train, g_train))

overzicht = pd.DataFrame(
    {f"fold {i}": pd.Series(y_train[va]).value_counts() for i, (_, va) in enumerate(folds, start=1)}
).reindex(KLASSEN).fillna(0).astype(int)
overzicht.loc["opnames"] = [len(set(g_train[va])) for _, va in folds]

for tr, va in folds:
    assert set(g_train[tr]).isdisjoint(g_train[va])  # geen opname in train én validatie
print("OK: geen overlap van source_file tussen train en validatie")
overzicht

## 4. Permutation importance per fold

In [ ]:
importances = []   # per fold: gemiddelde daling in macro-F1 per feature
fold_scores = []

for i, (tr, va) in enumerate(folds, start=1):
    model = clone(svm).fit(X_train.iloc[tr], y_train[tr])

    X_va, y_va = X_train.iloc[va], y_train[va]
    fold_scores.append(f1_score(y_va, model.predict(X_va), average="macro"))

    r = permutation_importance(model, X_va, y_va, scoring=SCORING,
                               n_repeats=N_REPEATS, random_state=SEED, n_jobs=-1)
    importances.append(pd.Series(r.importances_mean, index=FEATURES, name=f"fold {i}"))
    print(f"fold {i}: macro-F1 = {fold_scores[-1]:.3f}")

print(f"\nMacro-F1 over de folds: {np.mean(fold_scores):.3f} ± {np.std(fold_scores, ddof=1):.3f}")
imp_per_fold = pd.DataFrame(importances)

## 5. Tabel: importance per feature

- `importance_mean`: gemiddelde daling van de macro-F1 over de 5 folds als de feature door elkaar gehusseld wordt. Hoger = belangrijker.
- `importance_std`: spreiding over de folds. Is de std groter dan het gemiddelde, dan is de feature niet in elke fold belangrijk.
- Rond 0 of negatief: het model heeft de feature niet nodig (of haalt dezelfde informatie uit een andere feature).

In [ ]:
perm_tabel = pd.DataFrame({
    "importance_mean": imp_per_fold.mean(),
    "importance_std": imp_per_fold.std(),
}).sort_values("importance_mean", ascending=False)
perm_tabel.insert(0, "rang", range(1, len(perm_tabel) + 1))
perm_tabel.index.name = "feature"

pd.set_option("display.max_rows", 70)
perm_tabel.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 14))
t = perm_tabel.iloc[::-1]
ax.barh(t.index, t["importance_mean"], xerr=t["importance_std"],
        color="seagreen", ecolor="#999999", capsize=2)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Daling macro-F1 bij husselen (gemiddelde ± std over 5 folds)")
ax.set_title("Permutation importance – SVM, StratifiedGroupKFold op source_file")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

## 6. Samenvatting per featuregroep

In [ ]:
def groep(f):
    if f.startswith("mfcc_") and f.endswith("_mean"): return "mfcc_mean"
    if f.startswith("mfcc_") and f.endswith("_std"):  return "mfcc_std"
    if f.startswith("delta_"):                        return "delta_mean"
    return "spectraal (zcr/rms/centroid/rolloff)"

per_groep = (perm_tabel.assign(groep=[groep(f) for f in perm_tabel.index])
                       .groupby("groep")["importance_mean"]
                       .agg(["sum", "mean", "max", "count"])
                       .sort_values("sum", ascending=False))

print("Top 10:", ", ".join(perm_tabel.index[:10]))
print("Features met importance <= 0:", ", ".join(perm_tabel.index[perm_tabel["importance_mean"] <= 0]))
per_groep.round(4)

## Kanttekeningen (voor de taak 'Feature importance vergelijken en conclusie trekken')

- **Gecorreleerde features:** veel MFCC's hangen samen, net als `spectral_centroid`, `spectral_rolloff`, `zcr` en `mfcc_2`. Husselen we één feature, dan kan de SVM nog op een gecorreleerde feature leunen. Die feature lijkt dan minder belangrijk dan hij is. Een lage importance betekent dus niet automatisch dat de feature nutteloos is.
- **Kleine klassen:** de trainingsset heeft maar weinig seconden buzz en flight. De importances per fold schommelen daardoor flink (zie de std).
- **Ander model, andere importances:** `feature_selectie.ipynb` gebruikt een Random Forest op alleen de 16 kHz-opnames met 5 klassen. Verschillen met deze tabel kunnen daardoor komen.